# Recovery Reentry CPPI Floor v1

Notebook-owned experiment logic for a tighter drawdown-protection floor after `recovery_reentry_regime_drawdown_guard_v1` still missed the `15%` MaxDD target. Shared mechanics stay in `src/soxl_quant`; this notebook owns the lower-floor grid, candidate selection, walk-forward replay, poor-window diagnostics, and final readout.

Governance:
- Baseline: `recovery_reentry_regime_drawdown_guard_v1`
- Direct comparison: `recovery_reentry_anchor_confirm_v1`, `recovery_reentry_equity_drawdown_guard_v1`, and `recovery_reentry_regime_drawdown_guard_v1`
- Role: `diagnose`
- Changed layer: `execution`
- Target: strict OOS relative return >= +30pp, strict OOS MaxDD closer to 15%, and no material saved walk-forward regression
- Notebook rule: this `.ipynb` is the primary single-run conclusion artifact

External basis:
- Drawdown Measure in Portfolio Optimization: https://papers.ssrn.com/sol3/Delivery.cfm/SSRN_ID544742_code375170.pdf?abstractid=544742&mirid=1
- Theory of Constant Proportion Portfolio Insurance: https://www.sciencedirect.com/science/article/pii/016518899290043E
- Volatility Managed Portfolios: https://www.nber.org/papers/w22208
- Deep Momentum Networks: https://arxiv.org/abs/1904.04912


In [1]:
from __future__ import annotations

import itertools
import json
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path('/Users/rziiiii/.openclaw/workspace')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ['PYTHONPATH'] = 'src:.'

from src.soxl_quant.app.budget_counterfactual import (  # noqa: E402
    BullParticipationGuardConfig,
    EquityDrawdownControlConfig,
    RecoveryReentryGuardConfig,
    build_budget_usage_counterfactual,
    write_counterfactual_artifacts,
)
from src.soxl_quant.data import build_dataset, init_qlib  # noqa: E402
from src.soxl_quant.settings import QLIB_DATA_PATH  # noqa: E402

CARD = 'recovery_reentry_cppi_floor_v1'
CARD_DIR = ROOT / 'runtime' / 'execution' / 'card' / CARD
CARD_DIR.mkdir(parents=True, exist_ok=True)

STRICT_SLOW_PATH = ROOT / 'runtime/model/card/balanced_rule_residual_v19_reg/balanced_rule_residual_v19_reg_soxl_daily.csv'
STRICT_FAST_PATH = ROOT / 'runtime/model/card/balanced_rule_residual_v22/balanced_rule_residual_v22_soxl_daily.csv'
ANCHOR_JSON = ROOT / 'runtime/execution/card/recovery_reentry_anchor_confirm_v1/recovery_reentry_anchor_confirm_v1.json'
GLOBAL_GUARD_JSON = ROOT / 'runtime/execution/card/recovery_reentry_equity_drawdown_guard_v1/recovery_reentry_equity_drawdown_guard_v1.json'
REGIME_JSON = ROOT / 'runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/recovery_reentry_regime_drawdown_guard_v1.json'

WF_INPUTS = {
    'wf_2020': (
        ROOT / 'runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2020/balanced_rule_residual_v19_reg_input.csv',
        ROOT / 'runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2020/balanced_rule_residual_v22_input.csv',
    ),
    'wf_2022': (
        ROOT / 'runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2022/balanced_rule_residual_v19_reg_input.csv',
        ROOT / 'runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2022/balanced_rule_residual_v22_input.csv',
    ),
    'wf_2023_2024': (
        ROOT / 'runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2023_2024/balanced_rule_residual_v19_reg_input.csv',
        ROOT / 'runtime/model/card/v19reg_v22_budget_usage_v1/walkforward/inputs/wf_2023_2024/balanced_rule_residual_v22_input.csv',
    ),
}
REGIME_WF_JSONS = {
    key: ROOT / 'runtime/execution/card/recovery_reentry_regime_drawdown_guard_v1/walkforward' / key / f'recovery_reentry_regime_drawdown_guard_v1_{key}.json'
    for key in WF_INPUTS
}


def read_payload(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def variant_metric(path: Path, label: str) -> dict:
    return read_payload(path)['variants'][label]['test_proxy']


init_qlib(QLIB_DATA_PATH)
dataset = build_dataset()
price = dataset.handler.price_frame.droplevel('instrument')['price']
feature_frame = dataset.handler.feature_frame.droplevel('instrument').sort_index()
strict_slow = pd.read_csv(STRICT_SLOW_PATH)
strict_fast = pd.read_csv(STRICT_FAST_PATH)
wf_frames = {key: (pd.read_csv(slow), pd.read_csv(fast)) for key, (slow, fast) in WF_INPUTS.items()}

anchor_strict = variant_metric(ANCHOR_JSON, 'recovery_reentry_anchor_confirm_v1')
global_guard_strict = variant_metric(GLOBAL_GUARD_JSON, 'recovery_reentry_equity_drawdown_guard_v1')
regime_strict = variant_metric(REGIME_JSON, 'recovery_reentry_regime_drawdown_guard_v1')
regime_wf = {
    key: variant_metric(path, f'recovery_reentry_regime_drawdown_guard_v1_{key}')
    for key, path in REGIME_WF_JSONS.items()
}

print('anchor_strict', anchor_strict)
print('global_guard_strict', global_guard_strict)
print('regime_strict', regime_strict)


[72841:MainThread](2026-05-24 17:01:22,319) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[72841:MainThread](2026-05-24 17:01:23,046) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[72841:MainThread](2026-05-24 17:01:23,046) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[72841:MainThread](2026-05-24 17:01:25,762) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[72841:MainThread](2026-05-24 17:01:25,765) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[72841:MainThread](2026-05-24 17:01:25,770) INFO - qlib.timer - [log.py:127] - Time cost: 0.010s | Init data Done


anchor_strict {'rows': 1605, 'start': '2020-01-02', 'end': '2026-05-21', 'return': 1894.611414432602, 'sharpe': 1.1647912893695143, 'max_dd': 41.587206751681464, 'avg_pos': 0.46685181311301627, 'stress_avg_pos': 0.438732446228528, 'stress_days': 697, 'buy_and_hold_return': 816.5124997770644, 'relative_return': 1078.0989146555376, 'trades': 202}
global_guard_strict {'rows': 1605, 'start': '2020-01-02', 'end': '2026-05-21', 'return': 909.8458633564383, 'sharpe': 1.0750435873179458, 'max_dd': 34.36525136455344, 'avg_pos': 0.39040979173622337, 'stress_avg_pos': 0.3239584520879338, 'stress_days': 947, 'buy_and_hold_return': 816.5124997770644, 'relative_return': 93.3333635793739, 'trades': 334}
regime_strict {'rows': 1605, 'start': '2020-01-02', 'end': '2026-05-21', 'return': 2272.853019414519, 'sharpe': 1.2480221658238797, 'max_dd': 32.12017980873844, 'avg_pos': 0.4595798148626328, 'stress_avg_pos': 0.43573123179249773, 'stress_days': 618, 'buy_and_hold_return': 816.5124997770644, 'relative

In [2]:
bull_guard = BullParticipationGuardConfig(enabled=True)
reentry_guard = RecoveryReentryGuardConfig(
    enabled=True,
    min_target=0.35,
    max_target=0.55,
    max_base_target=0.25,
    min_mom_10=0.08,
    min_mom_21=0.12,
    min_relative_strength_21=1.0,
    min_vol_of_vol_21=0.10,
    max_vix_percentile_252=0.85,
    max_below_anchor_ratio_63=0.20,
    min_anchor_gap_180=0.75,
)


def config_from_values(
    start_drawdown: float,
    full_drawdown: float,
    min_target: float,
    max_anchor_gap_180: float,
    min_below_anchor_ratio_63: float,
    min_vix_percentile_252: float,
    max_mom_21: float,
) -> EquityDrawdownControlConfig:
    return EquityDrawdownControlConfig(
        enabled=True,
        start_drawdown=start_drawdown,
        full_drawdown=full_drawdown,
        min_target=min_target,
        sell_trade_threshold=0.0,
        sell_trade_lock_days=0,
        max_anchor_gap_180=max_anchor_gap_180,
        min_below_anchor_ratio_63=min_below_anchor_ratio_63,
        min_vix_percentile_252=min_vix_percentile_252,
        max_mom_21=max_mom_21,
    )


def config_from_row(row: pd.Series) -> EquityDrawdownControlConfig:
    return config_from_values(
        float(row['start_drawdown']),
        float(row['full_drawdown']),
        float(row['min_target']),
        float(row['max_anchor_gap_180']),
        float(row['min_below_anchor_ratio_63']),
        float(row['min_vix_percentile_252']),
        float(row['max_mom_21']),
    )


def config_payload(cfg: EquityDrawdownControlConfig) -> dict:
    return {
        'start_drawdown': cfg.start_drawdown,
        'full_drawdown': cfg.full_drawdown,
        'min_target': cfg.min_target,
        'sell_trade_threshold': cfg.sell_trade_threshold,
        'sell_trade_lock_days': cfg.sell_trade_lock_days,
        'max_anchor_gap_180': cfg.max_anchor_gap_180,
        'min_below_anchor_ratio_63': cfg.min_below_anchor_ratio_63,
        'min_vix_percentile_252': cfg.min_vix_percentile_252,
        'max_mom_21': cfg.max_mom_21,
    }


def build_result(slow: pd.DataFrame, fast: pd.DataFrame, cfg: EquityDrawdownControlConfig, *, profile: str, label: str, description: str):
    slow_label = 'balanced_rule_residual_v19_reg_current' if '_wf_' not in label else label.replace(CARD, 'balanced_rule_residual_v19_reg')
    fast_label = 'balanced_rule_residual_v22_current' if '_wf_' not in label else label.replace(CARD, 'balanced_rule_residual_v22')
    return build_budget_usage_counterfactual(
        slow,
        fast,
        price,
        slow_label=slow_label,
        fast_label=fast_label,
        composed_label=label,
        profile=profile,
        description=description,
        bull_guard=bull_guard,
        feature_frame=feature_frame,
        recovery_guard=reentry_guard,
        drawdown_control=cfg,
    )


def max_drawdown_event(daily: pd.DataFrame) -> dict:
    frame = daily.copy()
    frame['datetime'] = pd.to_datetime(frame['datetime'])
    price_df = price.rename('price').reset_index()
    price_df.columns = ['datetime', 'price']
    scored = frame.merge(price_df, on='datetime', how='inner')
    scored['price'] = scored['price'].astype(float)
    positions = scored['position'].to_numpy(dtype=float)
    returns = scored['price'].pct_change().fillna(0.0).to_numpy(dtype=float)
    nav = np.ones(len(scored))
    if len(scored) > 1:
        nav[1:] = np.cumprod(1.0 + positions[:-1] * returns[1:])
    peak = np.maximum.accumulate(nav)
    dd = nav / np.maximum(peak, 1e-12) - 1.0
    trough_idx = int(np.argmin(dd))
    peak_idx = int(np.argmax(nav[: trough_idx + 1])) if trough_idx >= 0 else 0
    return {
        'peak': scored['datetime'].iloc[peak_idx].date().isoformat(),
        'trough': scored['datetime'].iloc[trough_idx].date().isoformat(),
        'max_dd': abs(float(dd[trough_idx]) * 100.0),
        'peak_position': float(scored['position'].iloc[peak_idx]),
        'trough_position': float(scored['position'].iloc[trough_idx]),
    }


def candidate_row(candidate_id: int, cfg: EquityDrawdownControlConfig) -> dict:
    label = f'{CARD}_candidate_{candidate_id:03d}'
    result = build_result(
        strict_slow,
        strict_fast,
        cfg,
        profile=label,
        label=label,
        description='Strict-OOS notebook candidate for CPPI/TIPP-style lower drawdown floor.',
    )
    metric = result.payload['variants'][label]['test_proxy']
    row = {
        'candidate_id': candidate_id,
        **config_payload(cfg),
        'return': metric['return'],
        'relative_return': metric['relative_return'],
        'sharpe': metric['sharpe'],
        'max_dd': metric['max_dd'],
        'avg_pos': metric['avg_pos'],
        'trades': metric['trades'],
        'drawdown_control_days': int(result.daily['equity_drawdown_control'].sum()),
        'drawdown_regime_ready_days': int(result.daily['equity_drawdown_regime_ready'].sum()),
    }
    row['dd_improvement_vs_regime'] = regime_strict['max_dd'] - row['max_dd']
    row['return_giveback_vs_regime'] = max(regime_strict['return'] - row['return'], 0.0)
    row['passes_relative_gate'] = bool(row['relative_return'] >= 30.0)
    row['passes_15dd_gate'] = bool(row['max_dd'] <= 15.0)
    row['strict_score'] = (
        10.0 * max(row['max_dd'] - 15.0, 0.0)
        + max(30.0 - row['relative_return'], 0.0) * 5.0
        + 0.015 * row['return_giveback_vs_regime']
        + 0.02 * row['trades']
    )
    return row


In [3]:
grid_values = list(
    itertools.product(
        [0.12, 0.14],
        [0.16, 0.18, 0.20, 0.24],
        [0.00, 0.05, 0.10, 0.15, 0.25],
        [-0.10],
        [0.50, 0.70],
        [0.60, 0.70],
        [0.00, 0.05],
    )
)

rows = []
for candidate_id, values in enumerate(grid_values):
    rows.append(candidate_row(candidate_id, config_from_values(*values)))

screen = pd.DataFrame(rows).sort_values(
    ['passes_15dd_gate', 'passes_relative_gate', 'strict_score', 'max_dd'],
    ascending=[False, False, True, True],
)
screen_path = CARD_DIR / 'parameter_screen.csv'
screen.to_csv(screen_path, index=False)

eligible = screen[screen['passes_relative_gate'] & (screen['max_dd'] < regime_strict['max_dd'])].copy()
top = eligible.sort_values(['passes_15dd_gate', 'strict_score', 'max_dd'], ascending=[False, True, True]).head(12)
if top.empty:
    top = screen.sort_values(['strict_score', 'max_dd']).head(12)
top_path = CARD_DIR / 'parameter_screen_top.csv'
top.to_csv(top_path, index=False)

print(f'parameter_screen rows={len(screen)} target15={int(screen["passes_15dd_gate"].sum())} relative_pass={int(screen["passes_relative_gate"].sum())}')
print(top[['candidate_id', 'start_drawdown', 'full_drawdown', 'min_target', 'return', 'relative_return', 'max_dd', 'drawdown_control_days', 'strict_score']].to_string(index=False))


parameter_screen rows=320 target15=0 relative_pass=320
 candidate_id  start_drawdown  full_drawdown  min_target      return  relative_return   max_dd  drawdown_control_days  strict_score
          168            0.14           0.16        0.05 2271.783167      1455.270667 32.12018                    121    175.317846
          172            0.14           0.16        0.05 2271.783167      1455.270667 32.12018                    121    175.317846
          160            0.14           0.16        0.00 2410.985976      1594.473476 32.12018                    121    175.361798
          164            0.14           0.16        0.00 2410.985976      1594.473476 32.12018                    121    175.361798
          176            0.14           0.16        0.10 2292.496294      1475.983794 32.12018                    121    175.381798
          180            0.14           0.16        0.10 2292.496294      1475.983794 32.12018                    121    175.381798
          208        

In [4]:
wf_rows = []
for _, candidate in top.iterrows():
    cfg = config_from_row(candidate)
    candidate_id = int(candidate['candidate_id'])
    for wf_key, (slow_df, fast_df) in wf_frames.items():
        label = f'{CARD}_candidate_{candidate_id:03d}_{wf_key}'
        result = build_result(
            slow_df,
            fast_df,
            cfg,
            profile=label,
            label=label,
            description=f'Walk-forward replay for CPPI/TIPP floor candidate {candidate_id:03d} on {wf_key}.',
        )
        metric = result.payload['variants'][label]['test_proxy']
        base = regime_wf[wf_key]
        wf_rows.append(
            {
                'candidate_id': candidate_id,
                'wf': wf_key,
                'return': metric['return'],
                'relative_return': metric['relative_return'],
                'sharpe': metric['sharpe'],
                'max_dd': metric['max_dd'],
                'avg_pos': metric['avg_pos'],
                'trades': metric['trades'],
                'base_return': base['return'],
                'base_max_dd': base['max_dd'],
                'delta_return_vs_regime': metric['return'] - base['return'],
                'delta_max_dd_vs_regime': metric['max_dd'] - base['max_dd'],
                'wf_return_penalty': max(base['return'] - metric['return'], 0.0),
                'wf_dd_penalty': max(metric['max_dd'] - base['max_dd'], 0.0),
            }
        )

wf_screen = pd.DataFrame(wf_rows)
wf_path = CARD_DIR / 'walkforward_screen.csv'
wf_screen.to_csv(wf_path, index=False)

wf_summary = wf_screen.groupby('candidate_id').agg(
    wf_return_penalty=('wf_return_penalty', 'sum'),
    wf_dd_penalty=('wf_dd_penalty', 'sum'),
    min_delta_return_vs_regime=('delta_return_vs_regime', 'min'),
    max_delta_max_dd_vs_regime=('delta_max_dd_vs_regime', 'max'),
).reset_index()
wf2022 = wf_screen[wf_screen['wf'] == 'wf_2022'][['candidate_id', 'delta_return_vs_regime', 'delta_max_dd_vs_regime']].rename(
    columns={'delta_return_vs_regime': 'wf_2022_delta_return', 'delta_max_dd_vs_regime': 'wf_2022_delta_max_dd'}
)
selection = top.merge(wf_summary, on='candidate_id', how='left').merge(wf2022, on='candidate_id', how='left')
selection['wf_2022_ok'] = (selection['wf_2022_delta_return'] >= -3.0) & (selection['wf_2022_delta_max_dd'] <= 1.0)
selection['final_score'] = (
    10.0 * selection['max_dd'].sub(15.0).clip(lower=0.0)
    + selection['return_giveback_vs_regime'] * 0.015
    + selection['wf_return_penalty'] * 0.25
    + selection['wf_dd_penalty'] * 3.0
    + selection['passes_relative_gate'].map({True: 0.0, False: 500.0})
    + selection['wf_2022_ok'].map({True: 0.0, False: 100.0})
)
selection = selection.sort_values(['passes_15dd_gate', 'wf_2022_ok', 'final_score'], ascending=[False, False, True])
selection_path = CARD_DIR / 'selection_table.csv'
selection.to_csv(selection_path, index=False)

selected = selection.iloc[0]
selected_cfg = config_from_row(selected)
print(f'walkforward_screen rows={len(wf_screen)} wrote={wf_path.relative_to(ROOT)}')
print(selection[['candidate_id', 'return', 'relative_return', 'max_dd', 'passes_15dd_gate', 'wf_2022_ok', 'wf_return_penalty', 'wf_dd_penalty', 'final_score']].to_string(index=False))
print('selected_config', config_payload(selected_cfg))


walkforward_screen rows=36 wrote=runtime/execution/card/recovery_reentry_cppi_floor_v1/walkforward_screen.csv
 candidate_id      return  relative_return   max_dd  passes_15dd_gate  wf_2022_ok  wf_return_penalty  wf_dd_penalty  final_score
          160 2410.985976      1594.473476 32.12018             False        True                0.0            0.0   171.201798
          164 2410.985976      1594.473476 32.12018             False        True                0.0            0.0   171.201798
          176 2292.496294      1475.983794 32.12018             False        True                0.0            0.0   171.201798
          180 2292.496294      1475.983794 32.12018             False        True                0.0            0.0   171.201798
          208 2298.950911      1482.438411 32.12018             False        True                0.0            0.0   171.201798
          212 2298.950911      1482.438411 32.12018             False        True                0.0            0.0 

In [5]:
description = (
    'Notebook-selected CPPI/TIPP-style lower drawdown floor. It keeps anchor-confirmed recovery reentry and the same '
    'broken-anchor/high-volatility drawdown regime filters, but screens lower min_target values to test whether the '
    'remaining strict-OOS drawdown can move toward 15% without walk-forward overdefense.'
)
final_result = build_result(strict_slow, strict_fast, selected_cfg, profile=CARD, label=CARD, description=description)
final_artifacts = write_counterfactual_artifacts(final_result, CARD_DIR, price)
final_metric = final_result.payload['variants'][CARD]['test_proxy']
final_dd_event = max_drawdown_event(final_result.daily)
regime_ref_cfg = config_from_values(0.14, 0.20, 0.25, -0.10, 0.50, 0.60, 0.00)
regime_result = build_result(strict_slow, strict_fast, regime_ref_cfg, profile='regime_reference_replay', label='regime_reference_replay', description='Reference replay matching prior regime drawdown guard config.')
regime_dd_event = max_drawdown_event(regime_result.daily)

wf_artifact_rows = []
for wf_key, (slow_df, fast_df) in wf_frames.items():
    label = f'{CARD}_{wf_key}'
    result = build_result(
        slow_df,
        fast_df,
        selected_cfg,
        profile=label,
        label=label,
        description=f'Walk-forward replay for notebook-selected {CARD} on {wf_key}.',
    )
    paths = write_counterfactual_artifacts(result, CARD_DIR / 'walkforward' / wf_key, price)
    metric = result.payload['variants'][label]['test_proxy']
    wf_artifact_rows.append({'wf': wf_key, **metric, **{f'{kind}_path': str(path.relative_to(ROOT)) for kind, path in paths.items()}})

wf_artifacts = pd.DataFrame(wf_artifact_rows)
wf_artifacts_path = CARD_DIR / 'walkforward_summary.csv'
wf_artifacts.to_csv(wf_artifacts_path, index=False)

selection_payload = {
    'card': CARD,
    'selected_candidate_id': int(selected['candidate_id']),
    'selected_config': config_payload(selected_cfg),
    'anchor_strict': anchor_strict,
    'global_drawdown_guard_strict': global_guard_strict,
    'regime_drawdown_guard_strict': regime_strict,
    'selected_strict': final_metric,
    'regime_drawdown_event': regime_dd_event,
    'selected_drawdown_event': final_dd_event,
    'regime_wf': regime_wf,
    'selected_wf': wf_artifact_rows,
    'selection_rule': 'minimize strict-OOS drawdown gap to 15% with relative_return >= 30 and WF regression penalties.',
}
selection_json = CARD_DIR / 'selection.json'
selection_json.write_text(json.dumps(selection_payload, indent=2, default=lambda obj: obj.item() if hasattr(obj, 'item') else str(obj)), encoding='utf-8')
print('final_metric', final_metric)
print('final_dd_event', final_dd_event)
print('wrote', {kind: str(path.relative_to(ROOT)) for kind, path in final_artifacts.items()})
print('wrote', wf_artifacts_path.relative_to(ROOT), selection_json.relative_to(ROOT))


final_metric {'rows': 1605, 'start': '2020-01-02', 'end': '2026-05-21', 'return': 2410.9859762176493, 'sharpe': 1.2858241342240777, 'max_dd': 32.12017980873844, 'avg_pos': 0.4402787696715925, 'stress_avg_pos': 0.3683246346808821, 'stress_days': 662, 'buy_and_hold_return': 816.5124997770644, 'relative_return': 1594.4734764405848, 'trades': 208}
final_dd_event {'peak': '2021-04-05', 'trough': '2021-05-12', 'max_dd': 32.12017980873848, 'peak_position': 0.7186343814, 'trough_position': 0.8380035218}
wrote {'json': 'runtime/execution/card/recovery_reentry_cppi_floor_v1/recovery_reentry_cppi_floor_v1.json', 'daily': 'runtime/execution/card/recovery_reentry_cppi_floor_v1/recovery_reentry_cppi_floor_v1_soxl_daily.csv', 'review': 'runtime/execution/card/recovery_reentry_cppi_floor_v1/recovery_reentry_cppi_floor_v1.md'}
wrote runtime/execution/card/recovery_reentry_cppi_floor_v1/walkforward_summary.csv runtime/execution/card/recovery_reentry_cppi_floor_v1/selection.json


In [6]:
diag_dir = CARD_DIR / 'oos_poor_window_diagnostics'
diag_cmd = [
    sys.executable,
    'scripts/diagnose_oos_poor_windows.py',
    '--daily-csv',
    str(final_artifacts['daily']),
    '--profile',
    CARD,
    '--output-dir',
    str(diag_dir),
    '--oos-start',
    '2020-01-02',
    '--window-size',
    '63',
    '--target-excess',
    '30',
    '--target-max-dd',
    '15',
]
subprocess.run(diag_cmd, cwd=ROOT, check=True)
poor_windows = pd.read_csv(diag_dir / 'poor_windows.csv') if (diag_dir / 'poor_windows.csv').exists() else pd.DataFrame()

wf_lines = []
for row in wf_artifact_rows:
    wf_key = row['wf']
    base = regime_wf[wf_key]
    wf_lines.append(
        f"| `{wf_key}` | {base['return']:.2f}% / {base['max_dd']:.2f}% | "
        f"{row['return']:.2f}% / {row['max_dd']:.2f}% | "
        f"{row['return'] - base['return']:.2f}pp / {row['max_dd'] - base['max_dd']:.2f}pp |"
    )

poor_lines = []
for _, row in poor_windows.head(5).iterrows():
    poor_lines.append(
        f"- {row['start']}..{row['end']}: strategy={row['strategy_return']:.2f}%, "
        f"buyhold={row['buyhold_return']:.2f}%, excess={row['excess_return']:.2f}pp, max_dd={row['max_dd']:.2f}%"
    )
poor_text = '\n'.join(poor_lines) if poor_lines else '- no poor-window rows emitted'

decision = 'Diagnostic drop / no runtime promotion: no screened lower-floor candidate reached the 15% MaxDD target, and the selected strict-OOS MaxDD event is unchanged versus the regime baseline.'
if final_metric['max_dd'] <= 15.0 and final_metric['relative_return'] >= 30.0 and bool(selected['wf_2022_ok']):
    decision = 'Promotion candidate pending broader harness review.'

readout_lines = [
    '## Final Readout',
    '',
    'This notebook is the primary single-run conclusion artifact.',
    '',
    f"- Selected candidate: `{int(selected['candidate_id'])}`",
    f"- Selected config: `{json.dumps(config_payload(selected_cfg), sort_keys=True)}`",
    f"- Strict OOS regime baseline: `{regime_strict['return']:.2f}% / {regime_strict['sharpe']:.3f} / {regime_strict['max_dd']:.2f}% MaxDD / excess {regime_strict['relative_return']:.2f}pp`",
    f"- Strict OOS CPPI floor: `{final_metric['return']:.2f}% / {final_metric['sharpe']:.3f} / {final_metric['max_dd']:.2f}% MaxDD / excess {final_metric['relative_return']:.2f}pp`",
    f"- MaxDD event baseline: `{regime_dd_event['peak']} -> {regime_dd_event['trough']} ({regime_dd_event['max_dd']:.2f}%)`",
    f"- MaxDD event selected: `{final_dd_event['peak']} -> {final_dd_event['trough']} ({final_dd_event['max_dd']:.2f}%)`",
    f"- Decision: {decision}",
    '- Mechanism readout: lower protected floors improve return and `wf_2022`, but they do not touch the strict-OOS MaxDD because that event occurs in 2021 outside the current broken-anchor/high-vol downside filter.',
    '- Next valid hypothesis: diagnose the 2021 drawdown event separately instead of tightening the same regime floor.',
    '',
    '### Walk-Forward vs Regime Baseline',
    '',
    '| window | regime return / MaxDD | CPPI floor return / MaxDD | delta return / delta MaxDD |',
    '| --- | ---: | ---: | ---: |',
    *wf_lines,
    '',
    '### Worst Poor Windows',
    '',
    poor_text,
]
display(Markdown('\n'.join(readout_lines)))
print('single_run_conclusion: rendered above in notebook output')


[80108:MainThread](2026-05-24 17:02:12,161) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[80108:MainThread](2026-05-24 17:02:12,865) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.
[80108:MainThread](2026-05-24 17:02:12,865) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[80108:MainThread](2026-05-24 17:02:15,560) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done
[80108:MainThread](2026-05-24 17:02:15,560) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done
[80108:MainThread](2026-05-24 17:02:15,560) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


oos_diagnostic profile=recovery_reentry_cppi_floor_v1 windows=12 features=30 oos=2020-01-02..2026-05-21 output=/Users/rziiiii/.openclaw/workspace/runtime/execution/card/recovery_reentry_cppi_floor_v1/oos_poor_window_diagnostics


## Final Readout

This notebook is the primary single-run conclusion artifact.

- Selected candidate: `160`
- Selected config: `{"full_drawdown": 0.16, "max_anchor_gap_180": -0.1, "max_mom_21": 0.0, "min_below_anchor_ratio_63": 0.5, "min_target": 0.0, "min_vix_percentile_252": 0.6, "sell_trade_lock_days": 0, "sell_trade_threshold": 0.0, "start_drawdown": 0.14}`
- Strict OOS regime baseline: `2272.85% / 1.248 / 32.12% MaxDD / excess 1456.34pp`
- Strict OOS CPPI floor: `2410.99% / 1.286 / 32.12% MaxDD / excess 1594.47pp`
- MaxDD event baseline: `2021-04-05 -> 2021-05-12 (32.12%)`
- MaxDD event selected: `2021-04-05 -> 2021-05-12 (32.12%)`
- Decision: Diagnostic drop / no runtime promotion: no screened lower-floor candidate reached the 15% MaxDD target, and the selected strict-OOS MaxDD event is unchanged versus the regime baseline.
- Mechanism readout: lower protected floors improve return and `wf_2022`, but they do not touch the strict-OOS MaxDD because that event occurs in 2021 outside the current broken-anchor/high-vol downside filter.
- Next valid hypothesis: diagnose the 2021 drawdown event separately instead of tightening the same regime floor.

### Walk-Forward vs Regime Baseline

| window | regime return / MaxDD | CPPI floor return / MaxDD | delta return / delta MaxDD |
| --- | ---: | ---: | ---: |
| `wf_2020` | 113.89% / 26.14% | 113.89% / 26.14% | 0.00pp / 0.00pp |
| `wf_2022` | -17.13% / 24.22% | -13.22% / 16.03% | 3.90pp / -8.18pp |
| `wf_2023_2024` | 58.19% / 30.18% | 58.19% / 30.18% | 0.00pp / 0.00pp |

### Worst Poor Windows

- 2025-04-08..2025-07-09: strategy=38.93%, buyhold=223.03%, excess=-184.10pp, max_dd=11.50%
- 2025-04-21..2025-07-21: strategy=42.04%, buyhold=215.73%, excess=-173.69pp, max_dd=11.50%
- 2026-02-10..2026-05-11: strategy=25.60%, buyhold=200.63%, excess=-175.03pp, max_dd=6.42%
- 2026-02-05..2026-05-06: strategy=37.86%, buyhold=211.46%, excess=-173.59pp, max_dd=6.42%
- 2026-02-12..2026-05-13: strategy=24.65%, buyhold=191.98%, excess=-167.33pp, max_dd=6.42%

single_run_conclusion: rendered above in notebook output
